# LU Factorization Simulation

จำลองการทำงานของ `lu_factor`, `forward_solve`, `back_solve`  
จาก `src/algebra/matrix/matrix_solver.cpp`

## อัลกอริธึม: Doolittle LU Decomposition with Partial Pivoting

แยก matrix A (หลัง permutation P) ออกเป็น:
- **L**: lower-triangular, diagonal เป็น 1
- **U**: upper-triangular
- **P**: permutation (partial pivoting)

เงื่อนไข: **P·A = L·U**

### lu_factor loop structure
```
for k in 0..n-1:                    ← แต่ละ column k
    # Partial pivoting
    pivot_row = argmax(|lu[i][k]|, i=k..n-1)
    swap(lu[k], lu[pivot_row])
    swap(perm[k], perm[pivot_row])

    # Compute L column k (below diagonal)
    for i in k+1..n-1:
        lu[i][k] /= lu[k][k]         ← L[i][k] = A[i][k] / U[k][k]

    # Update Schur complement (U rows)
    for i in k+1..n-1:
        for j in k+1..n-1:
            lu[i][j] -= lu[i][k] * lu[k][j]
```

### forward_solve: L·y = P·b (L มี diagonal=1)
```
for i in 0..n-1:
    y[i] = b[perm[i]]
    for j in 0..i-1:
        y[i] -= L[i][j] * y[j]      ← ลบส่วนที่แก้ไปแล้ว
```

### back_solve: U·x = y
```
for i in n-1..0:
    x[i] = y[i]
    for j in i+1..n-1:
        x[i] -= U[i][j] * x[j]      ← ลบ known x
    x[i] /= U[i][i]                  ← หาร diagonal
```

In [1]:
from __future__ import annotations
import copy

kPivotTol = 1e-12


def fmt_lu(lu: list[list[float]], n: int) -> None:
    """แสดง combined LU matrix พร้อมบอกส่วน L และ U"""
    print("  LU matrix (L อยู่ใต้ diagonal, U อยู่บน+diagonal):")
    for i in range(n):
        row_str = ""
        for j in range(n):
            val = lu[i][j]
            if i == j:
                row_str += f"  [{val:>7.4g}]"  # diagonal = U[i][i]
            elif i > j:
                row_str += f"   {val:>7.4g} "  # below = L[i][j]
            else:
                row_str += f"   {val:>7.4g} "  # above = U[i][j]
        print(f"    row {i}:{row_str}")
    print()

In [2]:
def lu_factor(
    A: list[list[float]],
    n: int,
    verbose: bool = True,
) -> tuple[list[list[float]], list[int], float]:
    """
    Doolittle LU factorization with partial pivoting

    Returns
    -------
    lu : list[list[float]]
        Combined LU matrix (L below diagonal, U on/above)
    perm : list[int]
        Row permutation: perm[i] = original row index ที่ถูก swap มาที่ row i
    smallest_diag : float
        smallest |U[i][i]| → ถ้าใกล้ 0 แสดงว่า singular
    """
    lu = [row[:] for row in A]         # deep copy
    perm = list(range(n))              # identity permutation
    smallest_diag = float("inf")

    if verbose:
        print("=" * 60)
        print(f"LU Factorization (Doolittle): n={n}")
        print("=" * 60)
        fmt_lu(lu, n)

    for k in range(n):
        if verbose:
            print(f"{'─'*50}")
            print(f">>> k={k}: factorize column {k}")

        # ── Partial pivoting: หา row ที่มี |lu[i][k]| มากสุด ──────
        pivot_row = k
        max_val = abs(lu[k][k])

        if verbose:
            print(f"  Partial pivoting จาก row {k} ถึง {n-1}:")
            print(f"    row {k}: |lu[{k}][{k}]| = {max_val:.4g}")

        for i in range(k + 1, n):
            val = abs(lu[i][k])
            if verbose:
                arrow = " ← NEW MAX" if val > max_val else ""
                print(f"    row {i}: |lu[{i}][{k}]| = {val:.4g}{arrow}")
            if val > max_val:
                max_val = val
                pivot_row = i

        if pivot_row != k:
            lu[k], lu[pivot_row] = lu[pivot_row], lu[k]
            perm[k], perm[pivot_row] = perm[pivot_row], perm[k]
            if verbose:
                print(f"  SWAP row {k} ↔ row {pivot_row}  →  perm={perm}")
        else:
            if verbose:
                print(f"  ไม่ swap (pivot อยู่ที่ row {k})")

        # ── Track smallest diagonal ──────────────────────────────
        diag_val = abs(lu[k][k])
        smallest_diag = min(smallest_diag, diag_val)

        if verbose:
            print(f"  U[{k}][{k}] = {lu[k][k]:.4g}  (smallest_diag so far = {smallest_diag:.4g})")
            if diag_val < kPivotTol:
                print(f"  ⚠ diagonal ≈ 0 → singular!")

        # ── Compute L column k (Doolittle: L[i][k] = lu[i][k] / U[k][k]) ──
        if verbose:
            print(f"  Compute L[i][{k}] for i = {k+1} .. {n-1}:")

        for i in range(k + 1, n):
            if abs(lu[k][k]) < kPivotTol:
                break
            old_val = lu[i][k]
            lu[i][k] /= lu[k][k]
            if verbose:
                print(
                    f"    L[{i}][{k}] = lu[{i}][{k}] / lu[{k}][{k}]"
                    f" = {old_val:.4g} / {lu[k][k]:.4g} = {lu[i][k]:.4g}"
                )

        # ── Update Schur complement (U block) ─────────────────────
        if verbose:
            print(f"  Update Schur complement (rows {k+1}..{n-1}, cols {k+1}..{n-1}):")

        for i in range(k + 1, n):
            for j in range(k + 1, n):
                old_val = lu[i][j]
                delta = lu[i][k] * lu[k][j]
                lu[i][j] -= delta
                if verbose:
                    print(
                        f"    lu[{i}][{j}] = {old_val:.4g}"
                        f" - L[{i}][{k}]×U[{k}][{j}]"
                        f" = {old_val:.4g} - {lu[i][k]:.4g}×{lu[k][j]:.4g}"
                        f" = {lu[i][j]:.4g}"
                    )

        if verbose:
            fmt_lu(lu, n)

    if verbose:
        print("=" * 60)
        print("RESULT:")
        print(f"  perm = {perm}")
        print(f"  smallest_diag = {smallest_diag:.4g}")
        fmt_lu(lu, n)

    return lu, perm, smallest_diag

In [3]:
def forward_solve(
    lu: list[list[float]],
    perm: list[int],
    b: list[float],
    n: int,
    verbose: bool = True,
) -> list[float]:
    """
    Solve L·y = P·b  (L มี diagonal = 1, lower-triangular)

    b ถูก permute ด้วย perm ก่อน แล้วค่อย forward substitute
    """
    y = [0.0] * n

    if verbose:
        print("=" * 50)
        print("Forward Solve: L·y = P·b")
        print(f"  b = {b}")
        print(f"  perm = {perm}")
        pb = [b[perm[i]] for i in range(n)]
        print(f"  P·b = {pb}")
        print()

    for i in range(n):
        y[i] = b[perm[i]]          # apply permutation

        if verbose:
            print(f"  i={i}: y[{i}] = b[perm[{i}]] = b[{perm[i]}] = {y[i]:.4g}")
            print(f"         subtract L[{i}][j]·y[j] for j=0..{i-1}:")

        for j in range(i):
            L_ij = lu[i][j]         # L stored below diagonal
            sub = L_ij * y[j]
            if verbose:
                print(f"           j={j}: y[{i}] -= L[{i}][{j}]·y[{j}] = {L_ij:.4g}·{y[j]:.4g} = {sub:.4g}")
            y[i] -= sub

        if verbose:
            print(f"         → y[{i}] = {y[i]:.4g}")
            print()

    if verbose:
        print(f"  y = {[f'{v:.4g}' for v in y]}")
        print("=" * 50)

    return y


def back_solve(
    lu: list[list[float]],
    y: list[float],
    n: int,
    verbose: bool = True,
) -> list[float]:
    """
    Solve U·x = y  (U upper-triangular, diagonal ≠ 0)
    """
    x = [0.0] * n

    if verbose:
        print("=" * 50)
        print("Back Solve: U·x = y")
        print(f"  y = {y}")
        print()

    for i in range(n - 1, -1, -1):
        x[i] = y[i]

        if verbose:
            print(f"  i={i}: x[{i}] = y[{i}] = {x[i]:.4g}")
            print(f"         subtract U[{i}][j]·x[j] for j={i+1}..{n-1}:")

        for j in range(i + 1, n):
            U_ij = lu[i][j]         # U stored on/above diagonal
            sub = U_ij * x[j]
            if verbose:
                print(f"           j={j}: x[{i}] -= U[{i}][{j}]·x[{j}] = {U_ij:.4g}·{x[j]:.4g} = {sub:.4g}")
            x[i] -= sub

        if abs(lu[i][i]) > kPivotTol:
            x[i] /= lu[i][i]
            if verbose:
                print(f"         x[{i}] /= U[{i}][{i}] = {lu[i][i]:.4g}  → x[{i}] = {x[i]:.4g}")
        else:
            if verbose:
                print(f"         ⚠ U[{i}][{i}] ≈ 0, skip division")
        print()

    if verbose:
        print(f"  x = {[f'{v:.4g}' for v in x]}")
        print("=" * 50)

    return x

## Test Case 1: ระบบ 2×2 ปกติ

```
2x + 3y = 7
 x -  y = 1
```

คาดหวัง: x=2, y=1

In [4]:
A1 = [
    [2.0,  3.0],
    [1.0, -1.0],
]
b1 = [7.0, 1.0]

lu1, perm1, sd1 = lu_factor(A1, n=2)
y1 = forward_solve(lu1, perm1, b1, n=2)
x1 = back_solve(lu1, y1, n=2)

print(f"Solution: x={x1[0]:.4g}, y={x1[1]:.4g}")
assert abs(x1[0] - 2.0) < 1e-9 and abs(x1[1] - 1.0) < 1e-9, f"Expected x=2, y=1 but got {x1}"
print("✓ Test 1 passed")

LU Factorization (Doolittle): n=2
  LU matrix (L อยู่ใต้ diagonal, U อยู่บน+diagonal):
    row 0:  [      2]         3 
    row 1:         1   [     -1]

──────────────────────────────────────────────────
>>> k=0: factorize column 0
  Partial pivoting จาก row 0 ถึง 1:
    row 0: |lu[0][0]| = 2
    row 1: |lu[1][0]| = 1
  ไม่ swap (pivot อยู่ที่ row 0)
  U[0][0] = 2  (smallest_diag so far = 2)
  Compute L[i][0] for i = 1 .. 1:
    L[1][0] = lu[1][0] / lu[0][0] = 1 / 2 = 0.5
  Update Schur complement (rows 1..1, cols 1..1):
    lu[1][1] = -1 - L[1][0]×U[0][1] = -1 - 0.5×3 = -2.5
  LU matrix (L อยู่ใต้ diagonal, U อยู่บน+diagonal):
    row 0:  [      2]         3 
    row 1:       0.5   [   -2.5]

──────────────────────────────────────────────────
>>> k=1: factorize column 1
  Partial pivoting จาก row 1 ถึง 1:
    row 1: |lu[1][1]| = 2.5
  ไม่ swap (pivot อยู่ที่ row 1)
  U[1][1] = -2.5  (smallest_diag so far = 2)
  Compute L[i][1] for i = 2 .. 1:
  Update Schur complement (rows 2..1, col

## Test Case 2: ระบบ 3×3 — pivot swap จำเป็น

```
0·x + 2y + z = 5
3x -   y + 2z = 8
-x +   y -  z = -3
```

row 0 มี x=0 → partial pivoting จะ swap row 1 ขึ้นมา

In [5]:
A2 = [
    [ 0.0,  2.0,  1.0],
    [ 3.0, -1.0,  2.0],
    [-1.0,  1.0, -1.0],
]
b2 = [5.0, 8.0, -3.0]

lu2, perm2, sd2 = lu_factor(A2, n=3)
y2 = forward_solve(lu2, perm2, b2, n=3)
x2 = back_solve(lu2, y2, n=3)

print(f"Solution: x={x2[0]:.4g}, y={x2[1]:.4g}, z={x2[2]:.4g}")

# Verify: A2 @ x2 == b2
for i, eq_b in enumerate(b2):
    computed = sum(A2[i][j] * x2[j] for j in range(3))
    assert abs(computed - eq_b) < 1e-9, f"row {i}: A@x={computed:.4g} ≠ b={eq_b}"
print("✓ Test 2 passed (verified A@x == b)")

LU Factorization (Doolittle): n=3
  LU matrix (L อยู่ใต้ diagonal, U อยู่บน+diagonal):
    row 0:  [      0]         2          1 
    row 1:         3   [     -1]         2 
    row 2:        -1          1   [     -1]

──────────────────────────────────────────────────
>>> k=0: factorize column 0
  Partial pivoting จาก row 0 ถึง 2:
    row 0: |lu[0][0]| = 0
    row 1: |lu[1][0]| = 3 ← NEW MAX
    row 2: |lu[2][0]| = 1
  SWAP row 0 ↔ row 1  →  perm=[1, 0, 2]
  U[0][0] = 3  (smallest_diag so far = 3)
  Compute L[i][0] for i = 1 .. 2:
    L[1][0] = lu[1][0] / lu[0][0] = 0 / 3 = 0
    L[2][0] = lu[2][0] / lu[0][0] = -1 / 3 = -0.3333
  Update Schur complement (rows 1..2, cols 1..2):
    lu[1][1] = 2 - L[1][0]×U[0][1] = 2 - 0×-1 = 2
    lu[1][2] = 1 - L[1][0]×U[0][2] = 1 - 0×2 = 1
    lu[2][1] = 1 - L[2][0]×U[0][1] = 1 - -0.3333×-1 = 0.6667
    lu[2][2] = -1 - L[2][0]×U[0][2] = -1 - -0.3333×2 = -0.3333
  LU matrix (L อยู่ใต้ diagonal, U อยู่บน+diagonal):
    row 0:  [      3]        -1     

## Test Case 3: Verify P·A = L·U

ตรวจสอบว่า decomposition ถูกต้องจริง: สร้าง L และ U แยก แล้วคูณและเปรียบเทียบ

In [ ]:
def extract_L_U(lu: list[list[float]], n: int):
    """แยก L และ U จาก combined lu matrix"""
    L = [[0.0]*n for _ in range(n)]
    U = [[0.0]*n for _ in range(n)]
    for i in range(n):
        L[i][i] = 1.0                   # diagonal of L = 1 (unit lower triangular)
        for j in range(n):
            if j < i:
                L[i][j] = lu[i][j]      # below diagonal → L
            else:
                U[i][j] = lu[i][j]      # on/above diagonal → U
    return L, U


def mat_mul(A, B, n):
    """คูณ matrix n×n"""
    C = [[0.0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C


def permute_rows(A, perm, n):
    """สร้าง P·A จาก permutation"""
    return [A[perm[i]][:] for i in range(n)]


# Test กับ matrix 3×3
A3 = [
    [4.0, 3.0, 2.0],
    [8.0, 7.0, 1.0],
    [2.0, 4.0, 3.0],
]

lu3, perm3, _ = lu_factor(A3, n=3, verbose=False)
L3, U3 = extract_L_U(lu3, n=3)
PA3 = permute_rows(A3, perm3, n=3)
LU3 = mat_mul(L3, U3, n=3)

print("P·A:")
for row in PA3:
    print(f"  {[f'{v:.4g}' for v in row]}")

print("L·U:")
for row in LU3:
    print(f"  {[f'{v:.4g}' for v in row]}")

# ตรวจสอบว่า P·A ≈ L·U
for i in range(3):
    for j in range(3):
        assert abs(PA3[i][j] - LU3[i][j]) < 1e-9, \
            f"P·A[{i}][{j}]={PA3[i][j]:.4g} ≠ L·U[{i}][{j}]={LU3[i][j]:.4g}"

print("✓ P·A = L·U verified")

## Test Case 4: Singular Matrix Detection

```
x + y = 2
x + y = 3   ← inconsistent (no solution)
```

คาดหวัง: smallest_diag ≈ 0 หลัง factorization

In [ ]:
A4 = [
    [1.0, 1.0],
    [1.0, 1.0],
]
b4 = [2.0, 3.0]

lu4, perm4, sd4 = lu_factor(A4, n=2)
print(f"smallest_diag = {sd4:.4g}")
assert sd4 < kPivotTol, f"Expected singular, got smallest_diag={sd4}"
print("✓ Singular matrix correctly detected")

## สรุป: เปรียบเทียบ LU กับ Gauss สำหรับระบบเดียวกัน

LU เหมาะสำหรับเมื่อต้องแก้ Ax = b1, Ax = b2, ... หลาย b  
Gauss ทำทีเดียวรวม b แต่ต้อง redo ถ้าเปลี่ยน b

In [ ]:
A5 = [
    [3.0, 1.0, 2.0],
    [1.0, 4.0, 0.0],
    [2.0, 1.0, 3.0],
]

# Factor once
lu5, perm5, _ = lu_factor(A5, n=3, verbose=False)

# Solve สำหรับ 3 RHS ต่างกัน (ใช้ factorization เดิม)
RHS_list = [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
]

print("Solving A·X = I  (computing A⁻¹ using LU)")
inv_cols = []
for col_idx, b5 in enumerate(RHS_list):
    y5 = forward_solve(lu5, perm5, b5, n=3, verbose=False)
    x5 = back_solve(lu5, y5, n=3, verbose=False)
    inv_cols.append(x5)
    print(f"  column {col_idx}: {[f'{v:.4g}' for v in x5]}")

# A_inv คือ transpose ของ inv_cols
A_inv = [[inv_cols[j][i] for j in range(3)] for i in range(3)]
print("\nA⁻¹ =")
for row in A_inv:
    print(f"  {[f'{v:.4g}' for v in row]}")

# Verify A @ A_inv ≈ I
I = mat_mul(A5, A_inv, 3)
print("\nA @ A_inv (should be I):")
for row in I:
    print(f"  {[f'{v:.4g}' for v in row]}")

## สรุป Loop Structure

```
lu_factor(A, n):
│
├── for k in range(n):                         ← O(n)  pivot column
│   ├── for i in range(k+1, n):                ← O(n)  pivot search
│   │       find argmax |lu[i][k]|
│   ├── swap rows + perm                        ← O(n)  swap
│   ├── for i in range(k+1, n):                ← O(n)  compute L column
│   │       lu[i][k] /= lu[k][k]
│   └── for i in range(k+1, n):                ← O(n²) Schur complement
│           for j in range(k+1, n):
│               lu[i][j] -= lu[i][k] * lu[k][j]
│   Total: O(n³)
│
forward_solve(lu, perm, b, n):                 ← O(n²)
│   for i in range(n):
│       y[i] = b[perm[i]]
│       for j in range(i):
│           y[i] -= L[i][j] * y[j]
│
back_solve(lu, y, n):                          ← O(n²)
    for i in range(n-1, -1, -1):
        x[i] = y[i]
        for j in range(i+1, n):
            x[i] -= U[i][j] * x[j]
        x[i] /= U[i][i]
```

| Test | สถานการณ์ | Swap? | ผลลัพธ์ |
|------|-----------|-------|--------|
| 1 | 2×2 ปกติ | อาจมี | x=2, y=1 |
| 2 | 3×3 row=0 pivot=0 | ใช่ | verified A@x=b |
| 3 | Verify P·A=L·U | - | ✓ algebraically correct |
| 4 | Singular | - | smallest_diag≈0 |
| 5 | Multiple RHS | - | A⁻¹ computed |